Esta etapa silver toma la data de bronze de instrumentos y mejora la calidad de la información eliminando duplicados con un procesamiento incremental. Además este modulo es responsable de generar surrogate pk para el modelo de datos, qué se generarán a partir del nombre del ticket y la fecha de la información.

Inferimos el esquema del json para poder armar el modelo con el esquema qué necesitamos

In [0]:
ruta_origen = "iol_challenge.bronze.raw_ingestion"
ruta_destino = "iol_challenge.silver.deduped_instruments"

In [0]:
from pyspark.sql.functions import from_json, schema_of_json, col

data_sample = spark.read \
    .format("delta") \
    .table(ruta_origen) \
    .filter(col("data_type") == "instrument") \
    .withColumn("schema", schema_of_json(col("data"))) \
    .select("schema").first()[0]

In [0]:
data_sample

In [0]:
from pyspark.sql.functions import col, row_number, max as spark_max, length, md5, concat_ws, lit, explode, from_json
from pyspark.sql.window import Window
from delta.tables import DeltaTable

schema = """
STRUCT<simbolo: STRING,
date: STRING, 
Close: DOUBLE, 
High: DOUBLE, 
Low: DOUBLE, 
Open: DOUBLE, 
Volume: DOUBLE>
"""

df_origen_raw = spark.read \
    .format("delta") \
    .table(ruta_origen) \
    .filter(col("data_type") == "instrument") \
    .withColumn("parsed_dict", from_json(col("data"), schema)) \
    .withColumn("sk_instrument", md5(concat_ws(lit("||"), col("parsed_dict.date"), col("parsed_dict.simbolo")))) \
    .select(
        col("sk_instrument"),
        col("parsed_dict.*"), 
        col("dia"),
        col("anio"),
        col("mes"),
        col("timestamp_ejecucion"),
        col("errores_calidad"),
        col("tiene_errores_calidad"),
    )

# Tomamos la última versión ingestada de la transacción preferentemente sin errores de calidad
ventana_dedup = Window.partitionBy("sk_instrument").orderBy(col("tiene_errores_calidad").asc(), col("timestamp_ejecucion").desc())

df_lote_deduplicado = df_origen_raw \
    .withColumn("row_num", row_number().over(ventana_dedup)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \


if spark.catalog.tableExists(ruta_destino):
    tabla_destino = DeltaTable.forName(spark, ruta_destino)
    
    max_timestamp = tabla_destino.toDF() \
        .select(spark_max("timestamp_ejecucion")) \
        .collect()[0][0]
    
    if max_timestamp is not None:
        df_lote_filtrado = df_lote_deduplicado.filter(col("timestamp_ejecucion") >= max_timestamp)
    else:
        df_lote_filtrado = df_lote_deduplicado

    tabla_destino.alias("target") \
        .merge(
            df_lote_filtrado.alias("source"),
            "target.sk_instrument = source.sk_instrument"
        ) \
        .whenMatchedUpdateAll(
            condition="source.timestamp_ejecucion > target.timestamp_ejecucion"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

else:
    df_lote_deduplicado.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(ruta_destino)

In [0]:
%sql
SELECT * FROM iol_challenge.silver.deduped_instruments limit 10;


In [0]:
%sql
SELECT date, simbolo, COUNT(*) FROM iol_challenge.silver.deduped_instruments group by date, simbolo
    having count(*)>1;

Revisamos los datos de instrumentos ingestados por mes

In [0]:
%sql
SELECT COUNT(*),  month(date) FROM iol_challenge.silver.deduped_instruments
    GROUP BY month(date)